# DQL: Analytical Functions

Analytical functions calculate values across related rows while keeping row-level detail. They are powered by window specifications.


## Setup

Run this first. It creates a Spark session, finds the CSV files, loads them as DataFrames, and registers SQL temp views.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd(),
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or beside this notebook.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Window Setup

A window defines the rows each analytical function can see. This example partitions by department and orders by salary.


In [ ]:
dept_salary_window = Window.partitionBy("deptno").orderBy(F.col("sal").desc())
all_salary_window = Window.orderBy(F.col("sal").desc())


## ROW_NUMBER, RANK, and DENSE_RANK

`row_number` gives every row a unique sequence. `rank` leaves gaps after ties. `dense_rank` does not leave gaps.


In [ ]:
emp.select(
    "deptno", "empno", "ename", "job", "sal",
    F.row_number().over(dept_salary_window).alias("row_number_in_dept"),
    F.rank().over(dept_salary_window).alias("rank_in_dept"),
    F.dense_rank().over(dept_salary_window).alias("dense_rank_in_dept")
).orderBy("deptno", "row_number_in_dept").show(30)

spark.sql("""
SELECT deptno, empno, ename, job, sal,
       ROW_NUMBER() OVER (PARTITION BY deptno ORDER BY sal DESC) AS row_number_in_dept,
       RANK() OVER (PARTITION BY deptno ORDER BY sal DESC) AS rank_in_dept,
       DENSE_RANK() OVER (PARTITION BY deptno ORDER BY sal DESC) AS dense_rank_in_dept
FROM emp
ORDER BY deptno, row_number_in_dept
""").show(30)


## NTILE

`ntile(n)` splits ordered rows into `n` buckets. This example places employees into four salary bands.


In [ ]:
emp.select(
    "empno", "ename", "sal",
    F.ntile(4).over(all_salary_window).alias("salary_quartile")
).orderBy("salary_quartile", F.col("sal").desc()).show(30)

spark.sql("""
SELECT empno, ename, sal,
       NTILE(4) OVER (ORDER BY sal DESC) AS salary_quartile
FROM emp
ORDER BY salary_quartile, sal DESC
""").show(30)


## LAG and LEAD

`lag` looks at a previous row. `lead` looks at a following row. They are useful for comparisons.


In [ ]:
hire_window = Window.orderBy("hiredate")

emp.select(
    "empno", "ename", "hiredate", "sal",
    F.lag("sal").over(hire_window).alias("previous_salary"),
    F.lead("sal").over(hire_window).alias("next_salary")
).orderBy("hiredate").show(20)

spark.sql("""
SELECT empno, ename, hiredate, sal,
       LAG(sal) OVER (ORDER BY hiredate) AS previous_salary,
       LEAD(sal) OVER (ORDER BY hiredate) AS next_salary
FROM emp
ORDER BY hiredate
LIMIT 20
""").show()


## FIRST_VALUE and LAST_VALUE

These functions return values from the first or last row in the window frame. For `last_value`, define the frame through the end of the partition.


In [ ]:
dept_full_window = (
    Window.partitionBy("deptno")
    .orderBy(F.col("sal").desc())
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)

emp.select(
    "deptno", "empno", "ename", "sal",
    F.expr("first_value(ename)").over(dept_full_window).alias("highest_paid_in_dept"),
    F.expr("last_value(ename)").over(dept_full_window).alias("lowest_paid_in_dept")
).orderBy("deptno", F.col("sal").desc()).show(30)

spark.sql("""
SELECT deptno, empno, ename, sal,
       FIRST_VALUE(ename) OVER (
           PARTITION BY deptno ORDER BY sal DESC
           ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
       ) AS highest_paid_in_dept,
       LAST_VALUE(ename) OVER (
           PARTITION BY deptno ORDER BY sal DESC
           ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
       ) AS lowest_paid_in_dept
FROM emp
ORDER BY deptno, sal DESC
""").show(30)
